# FungMod Quickstart

Create entities, assemble a process-centered model, run through the current ODE engine, validate, plot, and save standardized outputs.

In [1]:
from pathlib import Path
import sys

import numpy as np

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fungal_model import Enzyme, Environment, ModelBuilder, ProcessRegistry, SimulationResult, WellMixedGeometry
from fungal_model.core.parameters import Parameter, ParameterSet
from fungal_model.core.simulation import SimulationEngine
from fungal_model.core.units import Q_
from fungal_model.core.validators import validate_mass_balance, validate_non_negative
from fungal_model.kinetics.surface_kinetics import PETSurfaceHydrolysisRateLaw
from fungal_model.substrates.pet import PETSubstrate, make_pet_parameter_set

In [2]:
pet = PETSubstrate(
    geometry_type="film",
    parameters=make_pet_parameter_set([
        Parameter("PET accessible surface area", "A_accessible", 0.1, "meter ** 2", 0.0, "Toy quickstart value.", "testing", "Notebook quickstart benchmark."),
    ]),
)

enzyme = Enzyme(
    name="toy PET-active hydrolase",
    enzyme_class="PETase-like hydrolase",
    target_bond_types=("ester",),
    target_substrate_names=("polyethylene terephthalate",),
    source="Toy quickstart enzyme metadata.",
)

environment = Environment(
    name="toy lab environment",
    temperature=Q_(303.15, "kelvin"),
    ph=Q_(7.0, "dimensionless"),
    water_activity=Q_(0.98, "dimensionless"),
    source="Toy quickstart environment metadata.",
)

geometry = WellMixedGeometry(
    volume=Q_(100.0, "milliliter"),
    surface_area=Q_(0.1, "meter ** 2"),
    source="Toy quickstart well-mixed geometry.",
)

In [3]:
rate_law = PETSurfaceHydrolysisRateLaw(
    pet=pet,
    enzyme="E",
    pet_mass="PET",
    adsorption_symbol="K_ads",
    surface_rate_symbol="k_surface",
    rate_units="kilogram / second",
    enzyme_units="mole / liter",
)
process = rate_law.as_generic_process(product_state="hydrolysate")
parameters = ParameterSet([
    Parameter("toy adsorption constant", "K_ads", 1.0, "liter / mole", 0.0, "Toy quickstart value.", "testing", "Notebook quickstart benchmark."),
    Parameter("toy surface catalysis constant", "k_surface", 1.0e-6, "kilogram / meter ** 2 / second", 0.0, "Toy quickstart value.", "testing", "Notebook quickstart benchmark."),
])

model = ModelBuilder(
    substrates=[pet],
    enzymes=[enzyme],
    environment=environment,
    geometry=geometry,
    process_library=ProcessRegistry([process]),
    requested_processes=("surface_catalysis",),
    parameters=parameters,
).assemble()

model.assembly_report.human_readable()

'Assembly report:\n    matched processes:\n        - PET surface hydrolysis through generic surface catalysis (surface_catalysis)\n    missing processes:\n        - none\n    missing parameters:\n        - none\n    incompatible units:\n        - none\n    incompatible mechanisms:\n        - none\n    warnings:\n        - none'

In [4]:
engine = SimulationEngine(
    reactions=[process.as_reaction()],
    parameters=parameters,
    species_units={"PET": "kilogram", "hydrolysate": "kilogram", "E": "mole / liter"},
    assumptions=list(process.assumptions),
)

raw_result = engine.simulate(
    initial_state={"PET": Q_(1.0e-4, "kilogram"), "hydrolysate": Q_(0.0, "kilogram"), "E": Q_(1.0, "mole / liter")},
    t_span=(Q_(0.0, "second"), Q_(10.0, "second")),
    t_eval=Q_(np.linspace(0.0, 10.0, 21), "second"),
)
validations = [
    validate_non_negative(raw_result),
    validate_mass_balance(raw_result, conserved_weights={"PET": 1.0, "hydrolysate": 1.0}),
]
result = SimulationResult.from_ode_result(
    raw_result,
    validation_results=validations,
    assembly_report=model.assembly_report,
    name="notebook_00_quickstart",
    label="toy",
)
result.save(ROOT / "outputs" / "notebook_00_quickstart", mass_balance_weights={"PET": 1.0, "hydrolysate": 1.0})
result.validation_report()

[{'name': 'non_negative',
  'passed': True,
  'message': 'All checked physical state variables remained non-negative within tolerance.',
  'details': {'minima': {'PET': 9.95e-05, 'hydrolysate': 0.0, 'E': 1.0},
   'relative_tolerance': 1e-09}},
 {'name': 'mass_balance',
  'passed': True,
  'message': 'Weighted conserved total remained constant within tolerance.',
  'details': {'closed_system': True,
   'included_species': ['PET', 'hydrolysate'],
   'units': 'kilogram',
   'initial_total': 0.0001,
   'final_total': 0.0001,
   'max_deviation': 1.3552527156068805e-20,
   'relative_deviation': 1.3552527156068805e-20,
   'relative_tolerance': 1e-09}}]